# 코랩으로 해보는 물 온도 맞추기
**버튼, 전압, 히터를 다루는 60분 실습**

물이 너무 차가우면 히터를 켜고, 온도가 맞으면 세기를 낮춥니다. 이 일을 컴퓨터가 하게 만들어 봅니다. 모든 실험은 화면 속 물통에서 합니다.

## 0. 시작하기 · 5분

1. Google Colab에서 이 파일을 열어요. 파일을 받았다면 **파일 → 노트북 열기 → 업로드**에서 선택하세요.
2. 아래 **실습 준비** 칸의 왼쪽 ▶를 눌러요. **준비 완료**가 나오면 시작할 수 있어요.
3. 위에서 아래로 한 칸씩 ▶를 눌러 결과를 보고, 실습에서 지정한 숫자만 바꿔 다시 눌러요.

한꺼번에 실행하려면 **런타임 → 모두 실행**을 눌러요. `NameError`는 앞의 준비 칸이 실행되지 않았을 때 나올 수 있어요. 이 경우 준비 칸의 ▶를 누르고 하던 칸을 다시 실행하세요.

**계산에 쓰인 코드가 궁금하면** 왼쪽 목차나 맨 아래의 **세부 코드 보기**로 가세요. 보고 싶은 부분을 고른 뒤 `show_code`에 체크하고 ▶를 누르면 실습에서 실제로 쓰는 코드가 나와요.

| 실습 | 시간 |
|---|---:|
| 0. 시작하기 | 5분 |
| 1. 온도를 읽고 전압을 보내기 | 10분 |
| 2. 버튼과 히터 켜고 끄기 | 8분 |
| 3. 켜는 시간으로 세기 바꾸기 | 10분 |
| 4. 물 온도를 60 °C로 맞추기 | 15분 |
| 5. 찬물이 들어온 뒤 다시 맞추기와 경보 | 12분 |
| **합계** | **60분** |


In [ ]:
# @title 실습 준비 (먼저 ▶를 누르세요)
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

BLUE = "#087EB6"
ORANGE = "#BF5719"
GREEN = "#158061"
DARK = "#364C5A"

plt.rcParams.update({
    "figure.figsize": (7.2, 3.3), "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25,
})


def adc_read(voltage, bits=10, vref=5.0):
    """가상 AI: 전압 → 정수 코드. 범위 밖 전압은 포화시킵니다."""
    max_code = 2**bits - 1
    return np.rint(np.clip(voltage, 0, vref) / vref * max_code).astype(int)

def adc_to_voltage(code_value, bits=10, vref=5.0):
    max_code = 2**bits - 1
    return np.clip(code_value, 0, max_code) / max_code * vref

def dac_write(code_value, bits=8, vref=5.0):
    """가상 AO: 최근접 정수 코드 → 전압. 증폭기·부하 특성은 생략."""
    max_code = 2**bits - 1
    return np.rint(np.clip(code_value, 0, max_code)) / max_code * vref

def debounce(samples, stable_samples=4):
    stable, candidate, count = 0, 0, 0
    result = []
    for value in samples:
        value = int(bool(value))
        if value == candidate:
            count += 1
        else:
            candidate, count = value, 1
        if count >= stable_samples:
            stable = candidate
        result.append(stable)
    return np.array(result, dtype=int)

def hysteresis_control(temperatures, low=49.0, high=51.0, initial=0):
    if low >= high:
        raise ValueError("low는 high보다 작아야 합니다.")
    state = int(bool(initial))
    output = []
    for temp in temperatures:
        if temp <= low:
            state = 1
        elif temp >= high:
            state = 0
        output.append(state)
    return np.array(output)

def make_pwm(duty=0.4, frequency=10.0, periods=3, ticks_per_period=100):
    if not 0 <= duty <= 1 or frequency <= 0:
        raise ValueError("duty는 0~1, frequency는 양수여야 합니다.")
    high_ticks = int(np.rint(duty * ticks_per_period))
    index = np.arange(periods * ticks_per_period)
    time = index / (frequency * ticks_per_period)
    signal = 5.0 * ((index % ticks_per_period) < high_ticks)
    return time, signal, high_ticks / ticks_per_period

def pwm_heater(frequency, duty=0.4, duration=24.0):
    # 100칸/주기로 PWM의 ON 시간을 충분히 분해합니다.
    dt_pwm = 1 / (frequency * 100)
    tt = np.arange(int(round(duration / dt_pwm)) + 1) * dt_pwm
    temperature = np.empty(len(tt)); temperature[0] = 25.0
    for k in range(len(tt) - 1):
        on = (k % 100) < int(np.rint(duty * 100))
        # 한 칸 동안 입력이 일정한 1차계의 정확한 이산화
        equilibrium = 25.0 + 40.0 * int(on)
        temperature[k+1] = equilibrium + (temperature[k] - equilibrium) * np.exp(-dt_pwm / 3)
    return tt, temperature

class PID:
    def __init__(self, kp=0.04, ki=0.003, kd=0.10, dt=0.2,
                 derivative_filter=1.0, anti_windup=True):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.dt, self.filter_tau = dt, derivative_filter
        self.anti_windup = anti_windup
        self.integral = 0.0  # 이미 Ki가 곱해진 I 항
        self.prev_y = None
        self.filtered_dy = 0.0

    def update(self, setpoint, measurement):
        error = setpoint - measurement
        p = self.kp * error
        raw_dy = 0.0 if self.prev_y is None else (measurement - self.prev_y) / self.dt
        alpha = self.dt / (self.filter_tau + self.dt)
        self.filtered_dy += alpha * (raw_dy - self.filtered_dy)
        d = -self.kd * self.filtered_dy
        self.prev_y = measurement

        candidate_i = self.integral + self.ki * error * self.dt
        candidate_u = p + candidate_i + d
        # 출력이 상한보다 높고 오차가 양수면, 적분 증가를 멈춥니다.
        # 출력이 하한보다 낮고 오차가 음수면, 적분 감소를 멈춥니다.
        deeper_saturation = ((candidate_u > 1 and error > 0) or
                             (candidate_u < 0 and error < 0))
        if not (self.anti_windup and deeper_saturation):
            self.integral = candidate_i
        u = float(np.clip(p + self.integral + d, 0, 1))
        return u, p, self.integral, d

def metrics(result, from_time=0, band=1.0):
    """평가는 센서 잡음이 아닌 실제 온도 사용. from_time 이후 지표."""
    mask = result['t'] >= from_time
    t = result['t'][mask]; error = result['target'][mask] - result['temp'][mask]
    outside = np.flatnonzero(np.abs(error) > band)
    if len(outside) == 0:
        settling = 0.0
    elif outside[-1] == len(t) - 1:
        settling = np.nan  # 기록 끝까지 미정착
    else:
        settling = t[outside[-1] + 1] - t[0]
    # 마지막 10초 평균. 모든 300초 사례에서 같은 기준을 사용.
    tail = result['t'] >= result['t'][-1] - 10
    return {"overshoot_C": float(max(0.0, np.max(-error))),
            "tail_bias_C": float(np.mean((result['target'] - result['temp'])[tail])),
            "IAE_Cs": float(np.sum(0.5 * (np.abs(error[:-1]) + np.abs(error[1:])) * np.diff(t))),
            "settling_s": float(settling)}

def print_metrics(name, result, from_time=0):
    m = metrics(result, from_time)
    settle_text = f"{m['settling_s']:.1f} s" if np.isfinite(m['settling_s']) else "미정착"
    print(f"{name}: 오버슈트 {m['overshoot_C']:.2f} °C / 종단 평균 편차 {m['tail_bias_C']:+.2f} °C / "
          f"IAE {m['IAE_Cs']:.1f} °C·s / ±1 °C정착 {settle_text}")

def simulate(kp=0.04, ki=0.003, kd=0.10, duration=300, dt=0.2,
             disturbance_at=None, disturbance=-10.0, noise_std=0.0,
             adc_bits=12, pwm_levels=None, anti_windup=True,
             derivative_filter=1.0, seed=42, initial_temp=25.0,
             initial_integral=0.0, fixed_output=None):
    """PID→평균 히터 출력→열모델→센서 전압→ADC의 폐루프.
    각 배열의 u[k]는 t[k]~t[k+1]에 적용됩니다. 마지막 u는 관측용입니다.
    """
    n_steps = int(round(duration / dt))
    time = np.arange(n_steps + 1) * dt
    target = np.full(len(time), 60.0)
    temp = np.empty(len(time)); temp[0] = initial_temp
    measured, command, applied, p_term, i_term, d_term = np.zeros((6, len(time)))
    disturbance_series = np.zeros(len(time))
    pid = PID(kp, ki, kd, dt, derivative_filter, anti_windup)
    pid.integral = initial_integral
    rng = np.random.default_rng(seed)
    decay = np.exp(-dt / 25.0)
    for k, now in enumerate(time):
        sensor_temp = temp[k] + rng.normal(0, noise_std)
        sensor_v = sensor_temp / 100.0 * 5.0
        measured[k] = float(adc_to_voltage(adc_read(sensor_v, adc_bits), adc_bits)) / 5 * 100
        command[k], p_term[k], i_term[k], d_term[k] = pid.update(target[k], measured[k])
        # pwm_levels=100이면 듀티비를 0~100칸으로 양자화합니다.
        applied[k] = (command[k] if pwm_levels is None else
                      np.rint(command[k] * pwm_levels) / pwm_levels)
        if fixed_output is not None:
            applied[k] = fixed_output
        disturbance_series[k] = (disturbance if disturbance_at is not None and
                                  now >= disturbance_at else 0.0)
        if k < n_steps:
            equilibrium = 25 + 80 * applied[k] + disturbance_series[k]
            temp[k+1] = equilibrium + (temp[k] - equilibrium) * decay
    return dict(t=time, target=target, temp=temp, measured=measured,
                u=applied, command=command, p=p_term, i=i_term, d=d_term,
                disturbance=disturbance_series)


def cooling_trial(auto_heat):
    # 냉각 전 60 °C를 유지하는 히터 세기는 43.75%입니다.
    normal_heat = 0.4375
    return simulate(kp=0.04, ki=0.003, kd=0, disturbance_at=150,
                    disturbance=-10, noise_std=0.15, adc_bits=12,
                    pwm_levels=100, seed=42, initial_temp=60,
                    initial_integral=normal_heat,
                    fixed_output=None if auto_heat else normal_heat)


def table(headings, rows):
    header = '| ' + ' | '.join(headings) + ' |'
    separator = '| ' + ' | '.join(['---'] * len(headings)) + ' |'
    body = ['| ' + ' | '.join(map(str, row)) + ' |' for row in rows]
    display(Markdown('\n'.join([header, separator, *body])))


def check_answer(answer, expected):
    if answer is None or (isinstance(answer, list) and any(x is None for x in answer)):
        print("아직 답을 넣지 않았어요. None을 숫자로 바꿔보세요.")
    elif answer == expected:
        print("✓ 맞았어요!")
    else:
        print("한 번 더 확인해 보세요.")


def temperatures_from_voltage(voltage):
    actual_temp = voltage * 20
    measurements = {
        bits: float(adc_to_voltage(adc_read(voltage, bits=bits), bits=bits) * 20)
        for bits in (4, 8)
    }
    return actual_temp, measurements


def show_adc(voltage):
    actual, measured = temperatures_from_voltage(voltage)
    print(f"두 경우 모두 같은 {voltage:.2f} V를 읽어요.")
    table(
        ["읽는 방법", "온도", "실제 온도와 차이"],
        [
            ["실제 물 온도", f"{actual:.2f} °C", "기준"],
            ["4비트 (16개 눈금)", f"{measured[4]:.2f} °C", f"{measured[4] - actual:+.2f} °C"],
            ["8비트 (256개 눈금)", f"{measured[8]:.2f} °C", f"{measured[8] - actual:+.2f} °C"],
        ],
    )


def check_adc_answer(answer_bits):
    actual, measured = temperatures_from_voltage(2.35)
    closest = min(measured, key=lambda bits: abs(measured[bits] - actual))
    check_answer(answer_bits, closest)


def show_dac(voltage):
    code = int(np.rint(np.clip(voltage, 0, 5) / 5 * 255))
    actual = float(dac_write(code))
    print(f"원하는 전압   {voltage:.3f} V")
    print(f"가장 가까운 눈금   {code}번")
    print(f"나온 전압   {actual:.4f} V")
    print(f"차이   {(actual - voltage) * 1000:+.2f} mV")


def show_button(stable_samples):
    raw = np.r_[np.zeros(4), [1, 0, 1, 0, 1, 1, 0], np.ones(7)].astype(int)
    filtered = debounce(raw, stable_samples=stable_samples)
    raw_rises = int(np.count_nonzero(np.diff(raw) == 1))
    filtered_rises = int(np.count_nonzero(np.diff(filtered) == 1))
    print(f"버튼을 누른 것으로 센 횟수   {raw_rises} → {filtered_rises}")
    fig, axes = plt.subplots(2, 1, figsize=(7, 3.1), sharex=True, layout="constrained")
    for ax, samples, name, color in zip(
            axes, (raw, filtered), ("Raw", "Filtered"), (ORANGE, BLUE)):
        ax.step(np.arange(len(samples)), samples, where="post", color=color, lw=2.5)
        ax.set(ylim=(-0.2, 1.2), yticks=[0, 1], ylabel=name)
    axes[-1].set_xlabel("Time (ms)")
    plt.show()


def heater_at_50(first_temperature):
    return int(hysteresis_control([first_temperature, 50])[-1])


def show_heater_memory():
    rows = []
    for first in (48, 52):
        output = heater_at_50(first)
        rows.append([f"{first} °C → 50 °C", f"{'켜짐' if output else '꺼짐'} ({output})"])
    table(["온도 순서", "50 °C일 때 히터"], rows)


def check_heater_answer(answer):
    check_answer(answer, [heater_at_50(48), heater_at_50(52)])


def show_pwm(duty):
    if not 0 <= duty <= 1:
        print("0부터 1까지의 숫자를 넣으세요. 예를 들어 50%는 0.5입니다.")
        return
    t, v, actual_duty = make_pwm(duty=duty, periods=1)
    average = float(v.mean())
    print(f"켜진 시간   {actual_duty:.0%}   |   평균 전압   {average:.1f} V")
    if np.isclose(average, 3):
        print("✓ 문제 3의 목표 3.0 V에 맞았어요!")

    fig, ax = plt.subplots(figsize=(6.8, 2.55), layout="constrained")
    end_ms = 100
    ax.step(np.r_[t * 1000, end_ms], np.r_[v, v[-1]], where="post", color=BLUE, lw=2.8)
    ax.axhline(average, color=ORANGE, ls="--", lw=1.8, label="Average")
    ax.set(xlabel="Time (ms)", ylabel="Voltage (V)", xlim=(0, end_ms), ylim=(-0.4, 5.7),
           xticks=[0, 50, 100], yticks=sorted({0, average, 5}))
    ax.legend(frameon=False, loc="upper right")
    plt.show()


def show_temperature(ki):
    result = simulate(kp=0.04, ki=ki, kd=0)
    print(f"마지막 물 온도   약 {result['temp'][-1]:.0f} °C")
    fig, ax = plt.subplots(layout="constrained")
    ax.axhline(60, color=GREEN, ls="--", lw=1.6, label="Goal")
    if ki != 0:
        p_only = simulate(kp=0.04, ki=0, kd=0)
        ax.plot(p_only["t"], p_only["temp"], color=ORANGE, lw=2.2, label="P")
    label = "P" if ki == 0 else "PI"
    ax.plot(result["t"], result["temp"], color=BLUE, lw=2.6, label=label)
    ax.set(xlim=(0, 140), ylim=(23, 65), yticks=[25, 40, 50, 60],
           xlabel="Time (s)", ylabel="Temperature (°C)")
    ax.legend(frameon=False, loc="lower right")
    plt.show()


def show_cooling(auto_heat):
    result = cooling_trial(auto_heat)
    print("히터를 자동으로 조절해요." if auto_heat else "히터를 처음 세기로 유지해요.")
    print(f"처음 물 온도   약 60 °C   |   마지막 물 온도   약 {result['temp'][-1]:.0f} °C")
    recovery = metrics(result, 150)["settling_s"]
    if np.isfinite(recovery):
        print(f"냉각 뒤 59–61 °C로 돌아와 유지   약 {recovery:.0f}초")
    else:
        print("냉각 뒤 59–61 °C로 돌아오지 못했어요.")

    fig, ax = plt.subplots(layout="constrained")
    ax.axhspan(59, 61, color="#DAF3EC", alpha=0.8, label="59–61 °C")
    ax.axhline(60, color=GREEN, ls="--", lw=1.3)
    ax.axvline(150, color=DARK, ls=":", lw=1.5)
    ax.plot(result["t"], result["temp"], color=BLUE if auto_heat else ORANGE, lw=2.8)
    ax.text(153, 61.45, "Cold water starts", color=DARK, fontsize=10)
    ax.set(xlim=(120, 300), ylim=(48.5, 62.7), xticks=[120, 150, 200, 250, 300],
           yticks=[50, 55, 60], xlabel="Time (s)", ylabel="Temperature (°C)")
    ax.legend(frameon=False, loc="lower left")
    plt.show()


def alarm_states(temperatures):
    state = 0
    result = []
    for temperature in temperatures:
        if temperature >= 65:
            state = 1
        elif temperature <= 63:
            state = 0
        result.append(state)
    return result


def check_alarm_answer(answer):
    check_answer(answer, alarm_states([62, 64, 65, 64, 63]))


print("준비 완료. 1-1부터 시작하세요!")


## 1. 온도를 읽고 전압을 보내기 · 10분

### 1-1. 같은 물, 다른 온도 눈금

자를 읽을 때 눈금이 촘촘하면 길이를 더 정확히 읽을 수 있어요. 온도계도 비슷해요. **실제 물 온도는 47.00 °C**, 온도 센서에서 오는 신호는 **2.35 V**예요. 아래 ▶를 눌러 **같은 2.35 V**를 두 가지 눈금으로 읽어봐요.

- **4비트**에는 눈금이 **16개**, **8비트**에는 **256개** 있어요.
- 표의 차이에서 **−는 실제보다 낮게**, **+는 실제보다 높게** 읽었다는 뜻이에요.

온도 센서의 전압을 눈금 번호로 읽는 일을 **아날로그 입력(AI)**이라고 하고, 이때 쓰는 장치를 **ADC**라고 해요.


In [ ]:
sensor_voltage = 2.35
show_adc(sensor_voltage)


#### 문제 1 · 어느 쪽이 실제 온도에 더 가까울까요?

위 표에서 실제 온도와의 차이가 더 작은 쪽을 찾아요. 아래 `None`을 **4 또는 8**로 바꿔 ▶를 눌러요.


In [ ]:
answer_bits = None  # 4 또는 8을 넣어요.
check_adc_answer(answer_bits)


### 1-2. 원하는 전압을 보내기

볼륨 조절기에 여러 칸이 있듯, 출력 전압도 미리 정해진 눈금 중 하나로 보낼 수 있어요. **3.010 V**를 원하면 가장 가까운 눈금인 **3.0196 V**가 나와요.

아래 ▶를 눌러 보고, 숫자 하나만 **3.010 → 3.000**으로 바꿔 다시 눌러요. 두 번째에는 원하는 값과 나온 값의 차이가 **0**이 되는지 보세요.

이렇게 눈금 번호를 전압으로 바꾸어 보내는 것이 **아날로그 출력(AO)**이고, 이때 쓰는 장치는 **DAC**예요.


In [ ]:
requested_voltage = 3.010  # 3.000으로 바꿔봐요.
show_dac(requested_voltage)


## 2. 버튼과 히터 켜고 끄기 · 8분

### 2-1. 버튼 한 번을 정확히 세기

손으로 버튼을 한 번 누르고 잠깐 그대로 둬요. 컴퓨터는 그 사이에 버튼 상태를 짧은 간격으로 여러 번 확인해요. 숫자 한 칸은 확인 한 번이에요.

버튼 안에는 금속 두 조각이 있어요. **떨어져 있으면 0, 붙으면 1**이에요. 손으로 버튼을 한 번 눌러도, 금속이 처음 맞닿는 아주 짧은 동안 떨어졌다 붙을 수 있어요. 그때 컴퓨터에는 0과 1이 여러 번 바뀐 것처럼 보여요.

- **바로 세면** 0에서 1로 올라갈 때마다 세어서, 이 예에서는 4번으로 보여요.
- **잠깐 기다리면** 1을 4번 연달아 읽었을 때 누른 것으로 정해요. 컴퓨터가 누름을 1회 기록해요. 버튼을 계속 눌러 1이 더 들어와도 기록은 1회예요.

▶를 누르고 위아래 그래프와 눌림 횟수를 비교해요. `stable_samples`를 **4에서 1로** 바꾸면, 1을 처음 읽자마자 바로 세기 시작해요. 결과가 어떻게 달라지나요?

버튼에서 0과 1을 읽어 들이는 일을 **디지털 입력(DI)**이라고 해요.



In [ ]:
stable_samples = 4  # 1로 바꿔봐요.
show_button(stable_samples)


### 2-2. 물이 차가울 때 켜고, 따뜻할 때 끄기

히터를 50 °C 근처에서 자꾸 켰다 끄지 않도록 두 기준을 정해요.

| 물 온도 | 히터 |
|---|---|
| 49 °C 이하 | 켜기 (1) |
| 49 °C와 51 °C 사이 (예: 50 °C) | 방금 상태 그대로 |
| 51 °C 이상 | 끄기 (0) |

▶를 눌러 **48 → 50 °C**와 **52 → 50 °C**를 비교해요. **50 °C라는 숫자는 같아도** 그 전에 켜져 있었는지에 따라 결과가 달라져요.

컴퓨터가 **켜기(1) 또는 끄기(0)**를 보내는 것이 **디지털 출력(DO)**이에요.


In [ ]:
show_heater_memory()


#### 문제 2 · 50 °C에서 히터가 켜져 있나요?

**48 → 50 °C**, **52 → 50 °C** 순서로 답을 적어요. 두 `None`을 **1 또는 0**으로 바꾸고 ▶를 눌러요.


In [ ]:
answer_do = [None, None]  # 켜짐은 1, 꺼짐은 0이에요.
check_heater_answer(answer_do)


## 3. 켜는 시간으로 세기 바꾸기 · 10분

전등을 아주 빨리 켰다 끌 때, 켜져 있는 시간이 길면 더 밝게 보여요. 화면 속 히터도 같은 방식으로 세기를 조절해요.

▶를 누르면 한 주기의 **절반** 동안 켜져요. 그래프에서 **5 V인 가로 구간**이 켜진 시간이고, **주황색 점선**은 평균이에요. 5 V를 절반 동안 켜면 **평균 2.5 V**예요.

이 방법을 **PWM**, 켜진 시간의 비율을 **듀티비**라고 해요. 코드에서는 50%를 **0.5**로 써요.

#### 문제 3 · 평균 3.0 V 만들기

`my_duty`의 **0.5**를 다른 숫자로 바꿔봐요. 목표는 **3.0 V**예요. 평균 전압은 **5 V × 켜진 비율**로 구할 수 있어요.


In [ ]:
my_duty = 0.5  # 바꾸기 전에는 2.5 V예요.
show_pwm(my_duty)


## 4. 물 온도를 60 °C로 맞추기 · 15분

샤워할 때 물이 차가우면 온수를 더 틀죠. 화면 속 컴퓨터도 물의 온도를 읽고 히터를 조절해요.

- **P**는 **지금** 목표보다 얼마나 차가운지 봐요.
- **I**는 조금 차가운 상태가 **계속**되면 출력을 조금씩 더 높여요.
- **D**는 물이 **빠르게** 뜨거워지고 있으면 출력을 미리 줄이는 방법이에요.

이번에는 **P만 쓰기**와 **P와 I를 같이 쓰기(PI)**를 비교해요. 세 방법을 같이 쓰면 **PID**라고 해요.

#### 문제 4 · 약 52 °C에 머무는 물을 목표 온도로 올리기

1. 아래 ▶를 눌러요. `my_ki`가 0이면 **P만** 쓰고, 마지막 온도는 **약 52 °C**예요.
2. `my_ki`만 **0.0 → 0.003**으로 바꿔 다시 눌러요. **PI**가 되어 마지막 온도가 **약 60 °C**에 가까워지는지 봐요.
3. 그래프에서 세로축은 **물 온도**, 초록 점선은 **목표 60 °C**예요. P와 PI 중 어느 선이 목표에 가까운지 이야기해 봐요.


In [ ]:
my_ki = 0.0  # 0.003으로 바꿔봐요.
show_temperature(my_ki)


## 5. 찬물이 들어온 뒤 다시 맞추기와 경보 · 12분

### 문제 5 · 찬물이 들어오면 히터가 어떻게 해야 할까요? (7분)

처음 물은 **60 °C**예요. **150초**에 찬물이 들어오기 시작해요.

1. 아래 ▶를 눌러요. `False`는 히터 세기 **고정**이에요. 마지막 물 온도가 얼마인지 봐요.
2. `False`를 **`True`**로 바꾸고 다시 눌러요. 이제 온도를 읽어 히터 세기를 **자동으로 조절**해요.
3. 그래프의 세로 점선이 **찬물 시작**, 초록 띠가 목표 주변 **59–61 °C**예요. 온도가 이 띠로 돌아오는지, 돌아오는 데 몇 초 걸리는지 봐요. 목표는 **60초 안에** 돌아와 그 범위에 머무르는 것이에요.

센서가 읽은 온도 숫자는 조금씩 흔들려요. 제어기는 그 숫자를 사용하고, 그래프는 화면 속 물의 실제 온도를 보여줘요.


In [ ]:
auto_heat = False  # True로 바꿔봐요.
show_cooling(auto_heat)


### 문제 6 · 너무 뜨거우면 경보등 켜기 (3분)

| 물 온도 | 경보등 |
|---|---|
| 65 °C 이상 | 켜기 (1) |
| 63 °C 이하 | 끄기 (0) |
| 63 °C와 65 °C 사이 (예: 64 °C) | 방금 상태 그대로 |

온도가 **62 → 64 → 65 → 64 → 63 °C** 순서로 변해요. 처음 세 칸은 같이 채워요. 62 °C에서 끄고 **0**, 64 °C에서 그대로 **0**, 65 °C에서 켜고 **1**이에요.

이제 마지막 두 `None`만 **0 또는 1**로 바꾸고 ▶를 눌러요.


In [ ]:
answer_alarm = [0, 0, 1, None, None]
check_alarm_answer(answer_alarm)


### 마무리 (2분)

같은 물 온도를 읽을 때 눈금이 촘촘하면 어떤 점이 좋아지나요? 찬물이 들어올 때 히터 세기를 자동으로 바꾸면 어떤 점이 달라졌나요? 두 질문에 한 문장씩 답해봐요.

## 예시 답안

풀이를 보려면 아래 칸의 `show_answers` 체크박스를 켜고 ▶를 눌러요. 체크박스가 보이지 않으면 **코드 표시**를 눌러 `False`를 `True`로 바꿔요. 새 실행 환경에서도 **이 칸 하나만 실행**하면 볼 수 있어요.


In [ ]:
# @title 예시 답안 표시
show_answers = False  # @param {type:"boolean"}

if show_answers:
    from IPython.display import Markdown, display

    display(Markdown(r"""
### 문제 1 · 8비트
실제 온도는 47.00 °C예요. 4비트는 46.67 °C로 **0.33 °C 낮게**, 8비트는 47.06 °C로 **0.06 °C 높게** 읽었어요. 8비트가 더 가까워요.
```python
answer_bits = 8
```

### 문제 2 · 1, 0
48 °C에서는 켜요. 50 °C가 되면 켜진 상태가 이어져요. 52 °C에서는 끄고, 50 °C가 되면 꺼진 상태가 이어져요.
```python
answer_do = [1, 0]
```

### 문제 3 · 60%
5 V의 60%는 3.0 V예요.
```python
my_duty = 0.6
```

### 문제 4 · I를 더하기
P만 쓰면 약 52 °C, I를 더하면 약 60 °C에 머물러요.
```python
my_ki = 0.003
```

### 문제 5 · 자동 조절 켜기
고정하면 마지막 물 온도는 약 50 °C예요. 자동이면 찬물이 들어와도 59–61 °C로 약 24초 만에 돌아와 유지돼요.
```python
auto_heat = True
```

### 문제 6 · 0, 0, 1, 1, 0
65 °C에서 켜고 64 °C에서도 켜진 채로 있다가, 63 °C에서 꺼요.
```python
answer_alarm = [0, 0, 1, 1, 0]
```
"""))
else:
    print("show_answers를 켜고 이 칸을 다시 실행하세요.")


## 더 알아보기 · 기본 실습을 마친 뒤

- **센서 눈금** 이 모형에서 $n$비트는 $2^n$개 눈금입니다. 0번을 0 V, 마지막 눈금을 5 V로 두고 가장 가까운 번호로 반올림합니다. 이 모형의 눈금 간격은 $5/(2^n-1)$ V입니다. 실제 ADC의 양자화 폭은 보통 $5/2^n$ V로 정의하며, 경계는 부품마다 다릅니다.
- **스위치의 두 기준** 히터나 경보등의 켜는 온도와 끄는 온도를 다르게 정하고, 그 사이에서 이전 상태를 유지하는 방법을 **히스테리시스**라고 합니다.
- **PWM과 물 온도** 한 주기의 켜진 비율을 $D$라고 하면 평균 전압은 $5D$ V입니다. 전압이 0 V 또는 5 V인 저항 히터의 평균 전력은 $D$에 비례합니다. 빠른 펄스가 들어와도 물은 천천히 온도가 변하므로 4·5절은 평균 히터 출력을 사용합니다.
- **가열조 모형** 주변은 25 °C이며, 열응답의 시정수는 25초입니다. 출력 $u$(0~1), 냉각 효과 $d$일 때 $25\,dT/dt = 25+80u+d-T$를 사용합니다. 0.2초마다 새 히터 출력을 계산합니다.
- **D와 출력 한계** 아래 PID 예제는 온도가 얼마나 빠르게 변하는지 계산해 D에 사용합니다. 작은 흔들림에 너무 민감해지지 않도록 필터를 사용합니다. 히터가 0% 또는 100%에 닿으면 I가 한계를 더 밀어붙이는 방향으로 커지지 않게 합니다. 이를 **anti-windup**이라고 합니다.

**실습 준비**를 실행한 뒤 아래 칸에서 `show_pid_demo`를 켜고 ▶를 누르면 P, PI, PID 그래프를 비교할 수 있어요. 추가 예제는 25 °C에서 시작해 150초에 냉각을 더합니다.


In [ ]:
# @title 더 알아보기 · P, PI, PID 비교와 냉각 예제
show_pid_demo = False  # @param {type:"boolean"}

if show_pid_demo:
    comparison = {
        "P": simulate(kp=0.04, ki=0, kd=0),
        "PI": simulate(kp=0.04, ki=0.003, kd=0),
        "PID": simulate(kp=0.04, ki=0.003, kd=0.10),
    }
    fig, ax = plt.subplots(layout="constrained")
    for (label, run), color in zip(comparison.items(), (ORANGE, BLUE, DARK)):
        ax.plot(run["t"], run["temp"], label=label, color=color, lw=2)
    ax.axhline(60, color=GREEN, ls="--", label="Goal")
    ax.set(xlim=(0, 120), xlabel="Time (s)", ylabel="Temperature (°C)")
    ax.legend(frameon=False)
    plt.show()

    solution = simulate(kp=0.06, ki=0.003, kd=0.10, disturbance_at=150, disturbance=-10,
                        noise_std=0.15, adc_bits=12, pwm_levels=100, seed=42)
    print_metrics("PID 예제 전체", solution)
    print_metrics("냉각 이후", solution, from_time=150)


### 더 알아보기 · 느린 PWM과 빠른 PWM

같은 40% 듀티비라도 10 Hz가 0.5 Hz보다 빠르게 깜빡입니다. `show_pwm_demo`를 켜고 ▶를 눌러 물 온도가 얼마나 덜 출렁이는지 확인해 보세요.


In [ ]:
# @title 더 알아보기 · PWM 빠르기와 온도 출렁임
show_pwm_demo = False  # @param {type:"boolean"}

if show_pwm_demo:
    fig, ax = plt.subplots(layout="constrained")
    for rate, color in [(0.5, ORANGE), (10, BLUE)]:
        time, temp = pwm_heater(rate)
        ax.plot(time, temp, label=f"{rate:g} Hz", color=color)
        print(f"{rate:g} Hz에서 마지막 6초간 온도 차이   {np.ptp(temp[time >= 18]):.2f} °C")
    ax.set(xlabel="Time (s)", ylabel="Temperature (°C)")
    ax.legend(frameon=False)
    plt.show()


## 세부 코드 보기 · 선택

앞에서 `show_adc(2.35)`를 실행하면 온도 표가 나왔어요. 아래에서 **ADC와 DAC**를 고르면 전압을 정수 번호로 바꾸고, 그 번호를 온도로 바꾸는 코드를 볼 수 있어요. **PWM**을 고르면 0 V와 5 V를 언제 내보내는지 나와요.

1. 맨 위 **실습 준비** 칸을 한 번 실행하세요. **준비 완료**가 나오면 됩니다.
2. 아래 칸에서 보고 싶은 부분을 고르고 `show_code`에 체크한 뒤 ▶를 누르세요. 체크박스가 보이지 않으면 `show_code = False`를 `True`로 바꾸세요.
3. **전체 준비 코드**를 고르면 그래프를 그리는 부분과 답안 확인에 쓰이는 함수까지 볼 수 있어요.


In [ ]:
# @title 세부 코드 펼치기
code_topic = "ADC와 DAC"  # @param ["ADC와 DAC", "버튼과 히터·경보", "PWM", "PID와 온도 모형", "전체 준비 코드"]
show_code = False  # @param {type:"boolean"}

if show_code:
    import ast
    import linecache
    from IPython import get_ipython
    from IPython.display import Code, display

    # 이미 실행한 준비 셀의 원문을 읽어, 실제 사용하는 코드를 그대로 보여 줍니다.
    def read_preparation_source():
        setup_function = globals().get("adc_read")
        if setup_function is None:
            return ""
        filename = setup_function.__code__.co_filename
        source = "".join(linecache.getlines(filename))
        if "def adc_read(" in source and "class PID:" in source:
            return source
        shell = get_ipython()
        history = shell.user_ns.get("In", []) if shell else []
        return next(
            (cell for cell in reversed(history)
             if cell.startswith("# @title 실습 준비") and "class PID:" in cell),
            "",
        )

    source = read_preparation_source()
    if "adc_read" not in globals():
        print("맨 위의 ‘실습 준비’에서 ▶를 누른 뒤 이 칸을 다시 실행하세요.")
    elif not source:
        print("이 실행 환경에서 준비 셀의 원문을 찾지 못했어요. 맨 위 ‘실습 준비’ 칸에서 코드를 펼치면 볼 수 있어요.")
    elif code_topic == "전체 준비 코드":
        display(Code(data=source, language="python"))
    else:
        topics = {
            "ADC와 DAC": (
                "adc_read", "adc_to_voltage", "dac_write",
                "temperatures_from_voltage", "show_adc", "show_dac",
            ),
            "버튼과 히터·경보": (
                "debounce", "show_button", "hysteresis_control",
                "heater_at_50", "show_heater_memory", "alarm_states",
            ),
            "PWM": ("make_pwm", "show_pwm", "pwm_heater"),
            "PID와 온도 모형": (
                "PID", "simulate", "metrics", "cooling_trial",
                "show_temperature", "show_cooling",
            ),
        }
        if code_topic not in topics:
            print("목록에서 보고 싶은 주제를 골라 주세요.")
        else:
            names = topics[code_topic]
            definitions = [
                ast.get_source_segment(source, node)
                for node in ast.parse(source).body
                if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in names
            ]
            if len(definitions) != len(names):
                print("함수 일부를 찾지 못했어요. ‘전체 준비 코드’를 골라 확인하세요.")
            else:
                display(Code(data="\n\n".join(definitions), language="python"))
else:
    print("코드가 궁금하면 주제를 고르고 show_code에 체크한 뒤 ▶를 누르세요.")
